# Day 16 — Decorators & Generators
 
**Week 3: Functions & Functional Programming**
 
Topics covered: decorator syntax (`@decorator`), decorators with arguments, `functools.wraps`, stacking decorators, generator functions (`yield`), generator expressions, `iter`/`next`, infinite generators.

### Q1. Your First Decorator
Write a decorator `shout` that takes a function, calls it, and converts its string return value to uppercase before returning it. Apply it to a function `greet(name)` that returns `"hello, {name}"`.

In [1]:
def shout(func):
    def wrapper(*args, **kwargs):
        result = func(*args, **kwargs)
        return result.upper()
    return wrapper

In [2]:
@shout
def greet(name):
    return f"hello, {name}"

In [3]:
print(greet("Chetan"))

HELLO, CHETAN


### Q2. Decorator that Adds Behavior Before and After
Write a decorator `logger` that prints `"Starting function..."` before calling the wrapped function and `"Function finished."` after. Apply it to any simple function.

In [4]:
def logger(func):
    def wrapper(*args, **kwargs):
        print("Starting function...")
        
        result = func(*args, **kwargs)
        
        print("Function finished.")
        return result
    
    return wrapper

In [5]:
@logger
def add(a, b):
    return a + b

In [6]:
print(add(3, 5))

Starting function...
Function finished.
8


### Q3. Decorator Preserving Arguments
Write a decorator `debug` that can wrap **any** function (regardless of how many positional/keyword arguments it takes) using `*args, **kwargs`, and prints the arguments received before calling the function.

In [7]:
def debug(func):
    def wrapper(*args, **kwargs):
        print(f"Arguments received:")
        print(f"args = {args}")
        print(f"kwargs = {kwargs}")
        
        return func(*args, **kwargs)
    
    return wrapper

In [8]:
@debug
def add(a, b):
    return a + b

In [9]:
print(add(2, 5))

Arguments received:
args = (2, 5)
kwargs = {}
7


### Q4. Using `functools.wraps`
Take your decorator from Q2 or Q3 and modify it to use `@functools.wraps(func)` on the inner wrapper function. Print the wrapped function's `__name__` and `__doc__` before and after adding `@wraps` to show the difference.

In [10]:
import functools

def debug(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        print("Calling function...")
        return func(*args, **kwargs)
    return wrapper
    

In [11]:
@debug
def greet(name):
    """Greets a person."""
    return f"Hello, {name}"

In [12]:
print(greet.__name__)
print(greet.__doc__)

greet
Greets a person.


### Q5. Timing Decorator
Write a decorator `timer` that measures and prints how long the wrapped function takes to execute (using the `time` module). Apply it to a function that does some repeated computation (e.g., a loop summing to a large number).

In [13]:
import time

def timer(func):
    def wrapper(*args, **kwargs):
        start = time.time()  # start time
        
        result = func(*args, **kwargs)
        
        end = time.time()    # end time
        
        print(f"Execution time: {end - start:.6f} seconds")
        
        return result
    
    return wrapper

In [14]:
@timer
def compute_sum(n):
    total = 0
    for i in range(n):
        total += i
    return total

In [15]:
print(compute_sum(10_000_00))

Execution time: 0.058993 seconds
499999500000


### Q6. Counting Calls Decorator
Write a decorator `count_calls` that tracks and prints how many times the decorated function has been called so far, each time it's called.

In [16]:
def count_calls(func):
    count = 0  # stores number of calls

    def wrapper(*args, **kwargs):
        nonlocal count
        count += 1
        
        print(f"Call #{count} to '{func.__name__}'")
        
        return func(*args, **kwargs)

    return wrapper

In [17]:
@count_calls
def multiply(a, b):
    return a * b

In [18]:
print(multiply(2, 3))
print(multiply(4, 5))
print(multiply(10, 2))

Call #1 to 'multiply'
6
Call #2 to 'multiply'
20
Call #3 to 'multiply'
20


### Q7. Decorator with Arguments
Write a decorator factory `repeat(n)` that returns a decorator which causes the wrapped function to be called `n` times in a row whenever it's invoked. Apply `@repeat(3)` to a function that prints a message.

In [19]:
def repeat(n):
    def decorator(func):
        def wrapper(*args, **kwargs):
            result = None
            for i in range(n):
                result = func(*args, **kwargs)
            return result
        return wrapper
    return decorator

In [20]:
@repeat(3)
def say_hello():
    print("Hello!")

In [21]:
say_hello()

Hello!
Hello!
Hello!


### Q8. Validating Arguments with a Decorator
Write a decorator `positive_only` that checks all positional arguments passed to the wrapped function are positive numbers; if any are not, it should raise a `ValueError` instead of calling the function.

In [22]:
def positive_only(func):
    def wrapper(*args, **kwargs):
        # check positional arguments
        for arg in args:
            if not isinstance(arg, (int, float)) or arg <= 0:
                raise ValueError(f"Invalid argument {arg}: all args must be positive numbers")
        
        return func(*args, **kwargs)
    
    return wrapper

In [23]:
@positive_only
def multiply(a, b):
    return a * b

In [24]:
print(multiply(3, 5))

15


### Q9. Stacking Multiple Decorators
Write two simple decorators, `bold` and `italic`, that wrap a function's string return value with `**` and `_` markers respectively (simulating markdown formatting). Apply both to a function `text()` that returns a plain string, and observe the order in which they're applied.

In [25]:
def bold(func):
    def wrapper(*args, **kwargs):
        result = func(*args, **kwargs)
        return f"**{result}**"
    return wrapper

In [26]:
def italic(func):
    def wrapper(*args, **kwargs):
        result = func(*args, **kwargs)
        return f"_{result}_"
    return wrapper

In [27]:
@bold
@italic
def text():
    return "Hello World"

In [28]:
print(text())

**_Hello World_**


### Q10. Caching Decorator (Manual `lru_cache`)
Without using `functools.lru_cache`, write your own decorator `memoize` that caches results of a function based on its arguments in a dictionary, so repeated calls with the same arguments return instantly from the cache.

In [29]:
def memoize(func):
    cache = {}

    def wrapper(*args, **kwargs):
        # Create a hashable key from args + kwargs
        key = (args, tuple(sorted(kwargs.items())))

        if key in cache:
            print("Returning from cache...")
            return cache[key]

        print("Computing result...")
        result = func(*args, **kwargs)
        cache[key] = result
        return result

    return wrapper

In [30]:
@memoize
def add(a, b):
    return a + b

In [31]:
print(add(2, 3))
print(add(2, 3))
print(add(4, 5))
print(add(2, 3))

Computing result...
5
Returning from cache...
5
Computing result...
9
Returning from cache...
5


### Q11. Class-Based Decorator (Optional/Bonus Style)
Write a decorator implemented as a class (with `__init__` and `__call__`) called `CallLogger` that logs every call to the wrapped function with its arguments. Apply it to a sample function.

In [32]:
class CallLogger:
    def __init__(self, func):
        self.func = func

    def __call__(self, *args, **kwargs):
        print(f"Calling '{self.func.__name__}'")
        print(f"args: {args}")
        print(f"kwargs: {kwargs}")

        result = self.func(*args, **kwargs)

        print(f"Finished '{self.func.__name__}'\n")
        return result

In [33]:
@CallLogger
def multiply(a, b):
    return a * b

In [34]:
print(multiply(3, 4))
print(multiply(10, 2))

Calling 'multiply'
args: (3, 4)
kwargs: {}
Finished 'multiply'

12
Calling 'multiply'
args: (10, 2)
kwargs: {}
Finished 'multiply'

20


### Q12. Your First Generator Function
Write a generator function `count_up_to(n)` that yields numbers from 1 up to `n`, one at a time. Use a `for` loop to print all values it generates.

In [35]:
def count_up_to(n):
    for i in range(1, n + 1):
        yield i

In [36]:
for num in count_up_to(5):
    print(num)

1
2
3
4
5


### Q13. Generator vs Regular Function (Memory Comparison)
Write a regular function `squares_list(n)` that returns a list of squares from 1 to `n`, and a generator function `squares_gen(n)` that yields the same squares one at a time. Use `sys.getsizeof()` to compare the memory size of the list version vs the generator object for `n = 1000000`.

In [37]:
import sys

def squares_list(n):
    return [i * i for i in range(1, n + 1)]

In [38]:
def squares_gen(n):
    for i in range(1, n + 1):
        yield i * i

In [39]:
n = 1_000_000

list_version = squares_list(n)
gen_version = squares_gen(n)

print("List size (bytes):", sys.getsizeof(list_version))
print("Generator size (bytes):", sys.getsizeof(gen_version))

List size (bytes): 8448728
Generator size (bytes): 224


### Q14. Using `next()` Manually
Create a generator object from `count_up_to(5)` (from Q12) and call `next()` on it manually 5 times, printing each result. Then call `next()` one more time and observe what exception is raised.


In [40]:
def count_up_to(n):
    for i in range(1, n + 1):
        yield i

In [41]:
gen = count_up_to(5)

In [42]:
print(next(gen))  # 1
print(next(gen))  # 2
print(next(gen))  # 3
print(next(gen))  # 4
print(next(gen))  # 5

1
2
3
4
5


In [43]:
print(next(gen))  # This will raise exception

StopIteration: 

In [44]:
StopIteration

StopIteration

### Q15. Generator Expression
Write a generator expression (similar syntax to a list comprehension but with parentheses) that produces the squares of numbers from 1 to 20. Convert it to a list to display the results, and separately demonstrate iterating it directly with a `for` loop.

In [45]:
gen = (x * x for x in range(1, 21))

In [46]:
squares_list = list(gen)
print(squares_list)

[1, 4, 9, 16, 25, 36, 49, 64, 81, 100, 121, 144, 169, 196, 225, 256, 289, 324, 361, 400]


### Q16. Infinite Generator
Write a generator function `infinite_counter(start=0)` that yields an infinite sequence of increasing numbers starting from `start`. Demonstrate it safely by only pulling the first 10 values (using something like `itertools.islice` or a manual loop with a `break`).

In [47]:
gen = (x * x for x in range(1, 21))

In [48]:
squares_list = list(gen)
print(squares_list)

[1, 4, 9, 16, 25, 36, 49, 64, 81, 100, 121, 144, 169, 196, 225, 256, 289, 324, 361, 400]


In [49]:
gen = (x * x for x in range(1, 21))
list(gen)

[1,
 4,
 9,
 16,
 25,
 36,
 49,
 64,
 81,
 100,
 121,
 144,
 169,
 196,
 225,
 256,
 289,
 324,
 361,
 400]

In [50]:
gen2 = (x * x for x in range(1, 21))

for value in gen2:
    print(value)

1
4
9
16
25
36
49
64
81
100
121
144
169
196
225
256
289
324
361
400


### Q17. Generator with Multiple `yield` Statements
Write a generator function `traffic_light()` that yields `"Red"`, then `"Yellow"`, then `"Green"`, in that order, then stops. Iterate over it with a `for` loop.

In [51]:
def traffic_light():
    yield "Red"
    yield "Yellow"
    yield "Green"

In [52]:
for light in traffic_light():
    print(light)

Red
Yellow
Green


### Q18. Sending Values into a Generator (`send`)
Write a generator function `running_total()` that uses `yield` to receive numbers sent to it (via `.send()`) and yields back the running total after each one. Demonstrate sending at least 4 values.

In [53]:
def running_total():
    total = 0
    
    while True:
        value = yield total   # receive value here
        if value is not None:
            total += value

In [54]:
gen = running_total()

In [55]:
print(next(gen))   # starts generator, outputs initial total = 0

0


In [56]:
print(gen.send(10))  # 10
print(gen.send(5))   # 15
print(gen.send(20))  # 35
print(gen.send(7))   # 42

10
15
35
42


### Q19. Generator Pipeline
Write three generator functions that form a pipeline: `read_numbers(n)` (yields 1 to n), `square_numbers(gen)` (yields the square of each number from the input generator), and `filter_even(gen)` (yields only even values from the input generator). Chain them together: `filter_even(square_numbers(read_numbers(10)))` and print the final results.

In [57]:
def read_numbers(n):
    for i in range(1, n + 1):
        yield i

In [58]:
def square_numbers(gen):
    for num in gen:
        yield num * num

In [59]:
def filter_even(gen):
    for num in gen:
        if num % 2 == 0:
            yield num

In [60]:
pipeline = filter_even(square_numbers(read_numbers(10)))

for value in pipeline:
    print(value)

4
16
36
64
100


### Q20. Fibonacci Generator
Write a generator function `fibonacci_gen()` that yields an infinite sequence of Fibonacci numbers. Use it to print the first 15 Fibonacci numbers only.

In [61]:
# Infinite Fibonacci generator

def fibonacci_gen():
    a, b = 0, 1
    while True:
        yield a
        a, b = b, a + b


# Use generator to print first 15 Fibonacci numbers
gen = fibonacci_gen()

for _ in range(15):
    print(next(gen))

0
1
1
2
3
5
8
13
21
34
55
89
144
233
377


### Q21. `yield from`
Write a generator function `flatten_gen(nested_list)` that uses `yield from` to recursively flatten an arbitrarily nested list and yield each individual element.

In [62]:
# Recursive generator using yield from

def flatten_gen(nested_list):
    for item in nested_list:
        if isinstance(item, list):
            # Recursively yield all items from sublist
            yield from flatten_gen(item)
        else:
            yield item


# Test case
data = [1, [2, 3], [4, [5, 6]], 7, [[8, 9], 10]]

print(list(flatten_gen(data)))

[1, 2, 3, 4, 5, 6, 7, 8, 9, 10]


### Q22. Decorator that Converts a Function into a Generator-Like Logger
Write a decorator `yield_results` that wraps a function returning a list, and instead causes calling code to be able to iterate over the results one at a time using a generator internally (i.e., the decorator converts the returned list into a generator before returning it from the wrapper).

In [63]:
# Decorator that converts list-returning function into generator-like output

def yield_results(func):
    def wrapper(*args, **kwargs):
        # Call original function (expects a list)
        result_list = func(*args, **kwargs)

        # Convert list into generator
        for item in result_list:
            yield item

    return wrapper


# Example usage
@yield_results
def get_numbers(n):
    return [i * i for i in range(n)]  # returns a list


# Test
gen = get_numbers(5)

for value in gen:
    print(value)

0
1
4
9
16


### Q23. Combining Decorators and Generators — Retry Logic
Write a decorator `retry(times)` that, if the wrapped function raises an exception, retries calling it up to `times` times before finally letting the exception propagate. Test it on a function that randomly raises an exception (using the `random` module) to see the retry behavior in action.

In [66]:
import random

# Decorator with retry logic
def retry(times):
    def decorator(func):
        def wrapper(*args, **kwargs):
            last_exception = None

            for attempt in range(1, times + 1):
                try:
                    print(f"Attempt {attempt}")
                    return func(*args, **kwargs)
                except Exception as e:
                    print(f"Failed on attempt {attempt}: {e}")
                    last_exception = e

            # After all retries fail, raise last exception
            raise last_exception

        return wrapper
    return decorator


# Function that randomly fails
@retry(times=5)
def unstable_function():
    if random.random() < 0.7:   # 70% chance of failure
        raise ValueError("Random failure occurred!")
    return "Success!"


# Test
result = unstable_function()
print("Result:", result)

Attempt 1
Failed on attempt 1: Random failure occurred!
Attempt 2
Result: Success!


### Q24. Generator-Based Pagination
Write a generator function `paginate(data, page_size)` that takes a list `data` and yields successive chunks (sub-lists) of size `page_size`. Test it with a list of 23 items and a page size of 5, and explain how the last (incomplete) page is handled.

In [67]:
# Generator for pagination

def paginate(data, page_size):
    for i in range(0, len(data), page_size):
        yield data[i:i + page_size]


# Test data (23 items)
data = list(range(1, 24))  # [1..23]

# Page size = 5
pages = paginate(data, 5)

for page_no, page in enumerate(pages, start=1):
    print(f"Page {page_no}: {page}")

Page 1: [1, 2, 3, 4, 5]
Page 2: [6, 7, 8, 9, 10]
Page 3: [11, 12, 13, 14, 15]
Page 4: [16, 17, 18, 19, 20]
Page 5: [21, 22, 23]


### Q25. Challenge — Lazy File-Like Line Processor
Simulate processing a large text source without loading it all into memory: create a list of 1000 strings representing "lines" of a file. Write a generator pipeline using at least two chained generator functions that (1) strips whitespace from each line, (2) filters out lines that are empty or shorter than 5 characters, and (3) yields only the first 10 valid results lazily (i.e., should not need to process all 1000 lines if 10 valid ones are found early — explain in a comment how generators make this efficient compared to processing the whole list upfront).

In [68]:
# Simulated large file (1000 lines)
lines = [f"   line {i}   " if i % 7 != 0 else "   " for i in range(1, 1001)]


# Generator 1: strip whitespace lazily
def strip_lines(iterable):
    for line in iterable:
        yield line.strip()


# Generator 2: filter invalid lines lazily
def filter_lines(iterable):
    for line in iterable:
        if len(line) >= 5:
            yield line


# Pipeline function (chaining generators)
def process(lines, limit=10):
    count = 0
    for line in filter_lines(strip_lines(lines)):
        yield line
        count += 1
        if count == limit:
            break


# Test
for item in process(lines, 10):
    print(item)


# ---------------------------------------------------------
# Why this is efficient:
# ---------------------------------------------------------
# 1. Generators are lazy: they process ONE item at a time.
# 2. strip_lines() does not create a full intermediate list.
# 3. filter_lines() also does not store results; it filters on the fly.
# 4. process() stops after 10 valid outputs using `break`.
#
# Unlike list-based processing, this avoids:
# - loading all 1000 processed lines into memory
# - unnecessary computation after 10 results are found
#
# So only enough input is processed to produce 10 valid outputs,
# making it efficient for large or infinite streams.

line 1
line 2
line 3
line 4
line 5
line 6
line 8
line 9
line 10
line 11
